# Variational Quantum Eigensolver (VQE)

This notebook implements the Variational Quantum Eigensolver (VQE) algorithm for quantum chemistry and molecular simulations.

### Pipeline Outline
1. **Environment Setup & Dependencies**: Installing Qiskit, Qiskit Nature, and PySCF.
2. **Molecular Hamiltonian Formulation**: Mapping molecular structure (e.g., $H_2$, $LiH$) to qubit operators (Jordan-Wigner / Parity transformation).
3. **Ansatz Formulation**: Parameterized quantum circuits (UCCSD / RealAmplitudes / EfficientSU2).
4. **Classical Optimization**: Classical minimizers (COBYLA, SLSQP, SPSA) evaluating ground state energy.
5. **Results & Potential Energy Surfaces**: Comparing VQE eigenvalues with Exact Diagonalization / Full CI.

In [1]:
# Setup environment, clone/pull latest repository, and install pinned dependencies
import os
import sys

!git config --global user.email "ruru-devs@users.noreply.github.com"
!git config --global user.name "ruru-devs"

if not os.path.exists("/content/Variational-Quantum-Eigensolver"):
    !git clone https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git /content/Variational-Quantum-Eigensolver
else:
    !cd /content/Variational-Quantum-Eigensolver && git pull

if "/content/Variational-Quantum-Eigensolver" not in sys.path:
    sys.path.insert(0, "/content/Variational-Quantum-Eigensolver")

%cd /content/Variational-Quantum-Eigensolver
!pip install -q -r requirements.txt


Cloning into '/content/Variational-Quantum-Eigensolver'...
remote: Enumerating objects: 60, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 60 (delta 19), reused 48 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (60/60), 483.62 KiB | 4.52 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/Variational-Quantum-Eigensolver
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [2]:
# Verify CPU runtime and mount Google Drive
import os, torch
from google.colab import drive

assert not torch.cuda.is_available(), "Runtime must be CPU! Please switch runtime type to CPU."
print("Runtime verified: CPU (CUDA available:", torch.cuda.is_available(), ")")

if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive", force_remount=False)
os.makedirs("/content/drive/MyDrive/mol-simul-VQE/results", exist_ok=True)
os.makedirs("/content/drive/MyDrive/mol-simul-VQE/checkpoints", exist_ok=True)
print("Google Drive mounted and mol-simul-VQE/ ready.")


Runtime verified: CPU (CUDA available: False )
Mounted at /content/drive
Google Drive mounted and mol-simul-VQE/ ready.


In [ ]:
# Run pytest with PYTHONPATH=. to verify all Phase 1 requirements
!PYTHONPATH=. pytest tests/ -v


============================= test session starts ==============================
platform linux -- Python 3.13.18, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/Variational-Quantum-Eigensolver
plugins: platformdirs-4.14.3, anyio-4.16.0, typeguard-4.6.0, langsmith-0.14.17
collected 6 items                                                              

tests/test_vqe.py::test_h2_pauli_terms_count PASSED                      [ 16%]
tests/test_vqe.py::test_sparse_hamiltonian_lowest_eigenvalue_equals_fci[H2-0.735] PASSED [ 33%]
tests/test_vqe.py::test_sparse_hamiltonian_lowest_eigenvalue_equals_fci[LiH-1.6] PASSED [ 50%]
tests/test_vqe.py::test_particle_number_and_norm_conservation PASSED     [ 66%]
tests/test_vqe.py::test_variational_principle_h2_and_lih PASSED          [ 83%]
tests/test_vqe.py::test_h2_chemical_accuracy PASSED                      [100%]

============================== 6 passed in 9.40s ===============================


In [ ]:
# Cell 1: H2 One Geometry (R = 0.735 A)
!cd /content/Variational-Quantum-Eigensolver && git pull
from src.sweep import run_single_geometry

res_h2 = run_single_geometry("H2", r=0.735, results_dir="/content/Variational-Quantum-Eigensolver/results")
print(f"H2 (R=0.735 A): E_VQE={res_h2['E_vqe']:.8f} Ha | E_FCI={res_h2['E_fci']:.8f} Ha | Error={res_h2['abs_error_mHa']:.4f} mHa | Iterations={res_h2['iterations']} | Wall time={res_h2['wall_s']:.2f}s")


remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 336 bytes | 336.00 KiB/s, done.
From https://github.com/ruru-devs/Variational-Quantum-Eigensolver
   a1a661b..ed0f1ed  main       -> origin/main
Updating a1a661b..ed0f1ed
Fast-forward
 tests/__init__.py | 0
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 tests/__init__.py
H2 (R=0.735 A): E_VQE=-1.13730604 Ha | E_FCI=-1.13730604 Ha | Error=0.0000 mHa | Iterations=3 | Wall time=0.15s


In [ ]:
# Cell 2: LiH One Geometry (R = 1.60 A)
from src.sweep import run_single_geometry
import shutil

res_lih = run_single_geometry("LiH", r=1.60, results_dir="/content/Variational-Quantum-Eigensolver/results")
print(f"LiH (R=1.60 A): E_VQE={res_lih['E_vqe']:.8f} Ha | E_FCI={res_lih['E_fci']:.8f} Ha | Error={res_lih['abs_error_mHa']:.4f} mHa | Iterations={res_lih['iterations']} | Wall time={res_lih['wall_s']:.2f}s")

# Git commit and push from Colab
!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 1: LiH one geometry" && git push origin main


LiH (R=1.60 A): E_VQE=-7.88209660 Ha | E_FCI=-7.88209660 Ha | Error=0.0000 mHa | Iterations=10 | Wall time=0.85s
[main 9f6e1d3] Phase 1: LiH one geometry
 2 files changed, 2 insertions(+)
 create mode 100644 results/lih_single_R=1.6000.csv
 create mode 100644 results/lih_single_R=1.6000.png
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 73.60 KiB | 12.27 MiB/s, done.
Total 5 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   acd120a..9f6e1d3  main -> main


In [ ]:
# Cell 3: H2O One Geometry (O-H = 0.96 A, H-O-H = 104.5 deg)
from src.sweep import run_single_geometry

# For H2O, sector dim is 225, n_params is 44. Precomputed sparse matrices evaluation is highly efficient.
res_h2o = run_single_geometry("H2O", r=0.96, angle_deg=104.5, results_dir="/content/Variational-Quantum-Eigensolver/results")
print(f"H2O (OH=0.96 A, 104.5 deg): E_VQE={res_h2o['E_vqe']:.8f} Ha | E_FCI={res_h2o['E_fci']:.8f} Ha | Error={res_h2o['abs_error_mHa']:.4f} mHa | Iterations={res_h2o['iterations']} | Wall time={res_h2o['wall_s']:.2f}s")

# Git commit and push from Colab
!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 1: H2O one geometry" && git push origin main


H2O (OH=0.96 A, 104.5 deg): E_VQE=-75.01297675 Ha | E_FCI=-75.01307689 Ha | Error=0.1001 mHa | Iterations=11 | Wall time=5.95s
[main 92cb8cf] Phase 1: H2O one geometry
 2 files changed, 2 insertions(+)
 create mode 100644 results/h2o_single_R=0.9600.csv
 create mode 100644 results/h2o_single_R=0.9600.png
Enumerating objects: 7, done.
Counting objects: 100% (7/7), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 63.31 KiB | 9.04 MiB/s, done.
Total 5 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   9f6e1d3..92cb8cf  main -> main


In [ ]:
# Cell 4: H2 Full Sweep (R = 0.30 to 2.50 A, step 0.05)
import numpy as np
from src.sweep import run_sweep

r_sweep_h2 = [round(float(x), 4) for x in np.arange(0.30, 2.50 + 1e-5, 0.05)]
df_h2 = run_sweep("H2", r_sweep_h2, results_dir="/content/Variational-Quantum-Eigensolver/results")

max_err_h2 = df_h2["abs_error_mHa"].max()
mean_err_h2 = df_h2["abs_error_mHa"].mean()
total_wall_h2 = df_h2["wall_s"].sum()
print(f"H2 Sweep Done: {len(df_h2)} points | Mean Error={mean_err_h2:.4f} mHa | Max Error={max_err_h2:.4f} mHa | Total Time={total_wall_h2:.2f}s")

# Git commit and push from Colab
!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 1: H2 full sweep" && git push origin main


H2 Sweep Done: 45 points | Mean Error=0.0000 mHa | Max Error=0.0000 mHa | Total Time=5.38s
[main fead101] Phase 1: H2 full sweep
 3 files changed, 47 insertions(+)
 create mode 100644 results/checkpoints/sweep_h2_checkpoint.json
 create mode 100644 results/h2_sweep.csv
 create mode 100644 results/h2_sweep.png
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 125.47 KiB | 9.65 MiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   92cb8cf..fead101  main -> main


In [ ]:
# Cell 5: LiH Full Sweep (R = 0.80 to 3.50 A, step 0.10)
import numpy as np
from src.sweep import run_sweep

r_sweep_lih = [round(float(x), 4) for x in np.arange(0.80, 3.50 + 1e-5, 0.10)]
df_lih = run_sweep("LiH", r_sweep_lih, results_dir="/content/Variational-Quantum-Eigensolver/results")

max_err_lih = df_lih["abs_error_mHa"].max()
mean_err_lih = df_lih["abs_error_mHa"].mean()
total_wall_lih = df_lih["wall_s"].sum()
print(f"LiH Sweep Done: {len(df_lih)} points | Mean Error={mean_err_lih:.4f} mHa | Max Error={max_err_lih:.4f} mHa | Total Time={total_wall_lih:.2f}s")

# Git commit and push from Colab
!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 1: LiH full sweep" && git push origin main


LiH Sweep Done: 28 points | Mean Error=0.0000 mHa | Max Error=0.0000 mHa | Total Time=28.42s
[main 9520a52] Phase 1: LiH full sweep
 3 files changed, 30 insertions(+)
 create mode 100644 results/checkpoints/sweep_lih_checkpoint.json
 create mode 100644 results/lih_sweep.csv
 create mode 100644 results/lih_sweep.png
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 132.99 KiB | 10.23 MiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   fead101..9520a52  main -> main


In [4]:
# Phase 2 Cell 1: H2O Full Symmetric O-H Stretch Sweep (R = 0.70 to 2.00 A, step 0.05, angle = 104.5 deg)
import numpy as np
from src.sweep import run_sweep

r_sweep_h2o = [round(float(x), 4) for x in np.arange(0.70, 2.00 + 1e-5, 0.05)]
print(f"Cost prediction from Phase 1 H2O (5.95s/pt): {len(r_sweep_h2o)} pts * 5.95s = {len(r_sweep_h2o)*5.95:.1f}s upper bound (faster with JAX autodiff + warm-start).")

df_h2o = run_sweep("H2O", r_sweep_h2o, angle_deg=104.5, use_jax=True, results_dir="/content/Variational-Quantum-Eigensolver/results")

max_err_h2o = df_h2o["abs_error_mHa"].max()
mean_err_h2o = df_h2o["abs_error_mHa"].mean()
total_wall_h2o = df_h2o["wall_s"].sum()
min_idx = df_h2o["E_vqe"].idxmin()
print(f"H2O Sweep Done: {len(df_h2o)} pts | Min E_VQE={df_h2o.loc[min_idx, 'E_vqe']:.8f} Ha at R={df_h2o.loc[min_idx, 'R']:.2f} A | Mean Err={mean_err_h2o:.4f} mHa | Max Err={max_err_h2o:.4f} mHa | Total Time={total_wall_h2o:.2f}s")

!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 2: H2O symmetric O-H stretch sweep" && git push origin main


Cost prediction from Phase 1 H2O (5.95s/pt): 27 pts * 5.95s = 160.7s upper bound (faster with JAX autodiff + warm-start).
H2O Sweep Done: 27 pts | Min E_VQE=-75.01966989 Ha at R=1.00 A | Mean Err=0.2950 mHa | Max Err=2.0449 mHa | Total Time=126.93s
[main 9000844] Phase 2: H2O symmetric O-H stretch sweep
 3 files changed, 29 insertions(+)
 create mode 100644 results/checkpoints/sweep_h2o_checkpoint.json
 create mode 100644 results/h2o_sweep.csv
 create mode 100644 results/h2o_sweep.png
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 142.82 KiB | 11.90 MiB/s, done.
Total 7 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 2 local objects.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   5d3c887..9000844  main -> main


In [5]:
# Phase 2 Cell 2: H2O 2D Bond-Angle & O-H Stretch Scan (O-H: 0.80-1.30 A step 0.10, angle: 90-130 deg step 10)
import numpy as np
from src.sweep import run_2d_sweep

r_grid_h2o = [round(float(x), 4) for x in np.arange(0.80, 1.30 + 1e-5, 0.10)]
ang_grid_h2o = [round(float(x), 1) for x in np.arange(90.0, 130.0 + 1e-5, 10.0)]

df_h2o_2d = run_2d_sweep(
    "H2O",
    r_grid_h2o,
    ang_grid_h2o,
    use_jax=True,
    results_dir="/content/Variational-Quantum-Eigensolver/results",
)

min_vqe_idx = df_h2o_2d["E_vqe"].idxmin()
min_fci_idx = df_h2o_2d["E_fci"].idxmin()
max_err_2d = df_h2o_2d["abs_error_mHa"].max()
mean_err_2d = df_h2o_2d["abs_error_mHa"].mean()
total_wall_2d = df_h2o_2d["wall_s"].sum()

print(
    f"H2O 2D Grid Done: {len(df_h2o_2d)} pts | "
    f"VQE Min={df_h2o_2d.loc[min_vqe_idx, 'E_vqe']:.8f} Ha at (R={df_h2o_2d.loc[min_vqe_idx, 'R']:.2f} A, {df_h2o_2d.loc[min_vqe_idx, 'angle_deg']:.0f} deg) | "
    f"FCI Min={df_h2o_2d.loc[min_fci_idx, 'E_fci']:.8f} Ha at (R={df_h2o_2d.loc[min_fci_idx, 'R']:.2f} A, {df_h2o_2d.loc[min_fci_idx, 'angle_deg']:.0f} deg) | "
    f"Mean Err={mean_err_2d:.4f} mHa | Max Err={max_err_2d:.4f} mHa | Total Time={total_wall_2d:.2f}s"
)

!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Phase 2: H2O 2D bond-angle sweep" && git push origin main


H2O 2D Grid Done: 30 pts | VQE Min=-75.02146685 Ha at (R=1.00 A, 100 deg) | FCI Min=-75.02156705 Ha at (R=1.00 A, 100 deg) | Mean Err=0.2073 mHa | Max Err=0.7527 mHa | Total Time=138.24s
[main da3827e] Phase 2: H2O 2D bond-angle sweep
 4 files changed, 32 insertions(+)
 create mode 100644 results/checkpoints/sweep_h2o_2d_checkpoint.json
 create mode 100644 results/h2o_2d_error.png
 create mode 100644 results/h2o_2d_sweep.csv
 create mode 100644 results/h2o_2d_sweep.png
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 369.19 KiB | 13.67 MiB/s, done.
Total 8 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/ruru-devs/Variational-Quantum-Eigensolver.git
   9000844..da3827e  main -> main


In [8]:
# 3A: N2 One Geometry (R = 1.098 A) with sparse adjoint gradient
!cd /content/Variational-Quantum-Eigensolver && git pull origin main
import importlib, src.vqe, src.sweep
importlib.reload(src.vqe)
importlib.reload(src.sweep)
from src.sweep import run_single_geometry

res_n2 = run_single_geometry("N2", r=1.098, use_adjoint=True, results_dir="/content/Variational-Quantum-Eigensolver/results")
print(
    f"N2 (R=1.098 A): E_VQE={res_n2['E_vqe']:.8f} Ha | E_FCI={res_n2['E_fci']:.8f} Ha | "
    f"Error={res_n2['abs_error_mHa']:.4f} mHa | Iterations={res_n2['iterations']} | Wall time={res_n2['wall_s']:.2f}s"
)

!cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Add N2 single-geometry VQE benchmark at equilibrium" && git push origin main


remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 8 (delta 6), reused 7 (delta 5), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.18 KiB | 558.00 KiB/s, done.
From https://github.com/ruru-devs/Variational-Quantum-Eigensolver
 * branch            main       -> FETCH_HEAD
   3444c80..5618a72  main       -> origin/main
Updating 3444c80..5618a72
Fast-forward
 README.md    | 18 ++++++++++++++++
 src/sweep.py |  8 +++++---
 src/vqe.py   | 67 ++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
 3 files changed, 90 insertions(+), 3 deletions(-)
N2 (R=1.098 A): E_VQE=-107.65056190 Ha | E_FCI=-107.65270379 Ha | Error=2.1419 mHa | Iterations=12 | Wall time=34.19s
[main a0a2d7b] Add N2 single-geometry VQE benchmark at equilibrium
 2 files changed, 2 insertions(+)
 create mode 100644 results/n2_single_R=1.0980.csv
 create mode 100644 results/n2_single_R=1.0980.png
Enumerating objects: 7

In [12]:
# 3B: N2 Full Sweep (R = 0.90 to 2.20 A, step 0.10) — resumable in batches
!cd /content/Variational-Quantum-Eigensolver && git pull origin main
import importlib, numpy as np, src.sweep
importlib.reload(src.sweep)
from src.sweep import run_sweep

r_sweep_n2 = [round(float(x), 4) for x in np.arange(0.90, 2.20 + 1e-5, 0.10)]
print(f"Cost prediction from 3A (34.19s/pt): {len(r_sweep_n2)} pts * 34.19s = {len(r_sweep_n2)*34.19:.1f}s (~8.0 min) upper bound.")

df_n2 = run_sweep(
    "N2",
    r_sweep_n2,
    use_adjoint=True,
    results_dir="/content/Variational-Quantum-Eigensolver/results",
    max_points=4,
)

print(f"N2 Sweep Progress: {len(df_n2)}/{len(r_sweep_n2)} points completed.")
if len(df_n2) == len(r_sweep_n2):
    max_err_n2 = df_n2["abs_error_mHa"].max()
    mean_err_n2 = df_n2["abs_error_mHa"].mean()
    total_wall_n2 = df_n2["wall_s"].sum()
    print(f"N2 Sweep Complete: Mean Err={mean_err_n2:.4f} mHa | Max Err={max_err_n2:.4f} mHa | Total Time={total_wall_n2:.2f}s")
    !cd /content/Variational-Quantum-Eigensolver && git add results/ && git commit -m "Add N2 dissociation curve sweep with sparse adjoint VQE" && git push origin main


From https://github.com/ruru-devs/Variational-Quantum-Eigensolver
 * branch            main       -> FETCH_HEAD
Already up to date.
Cost prediction from 3A (34.19s/pt): 14 pts * 34.19s = 478.7s (~8.0 min) upper bound.
Resuming N2 sweep: 12 already computed, 2 remaining.
N2 Sweep Progress: 14/14 points completed.
N2 Sweep Complete: Mean Err=4.5045 mHa | Max Err=11.8024 mHa | Total Time=444.70s
[main 02ff1bc] Add N2 dissociation curve sweep with sparse adjoint VQE
 3 files changed, 16 insertions(+)
 create mode 100644 results/checkpoints/sweep_n2_checkpoint.json
 create mode 100644 results/n2_sweep.csv
 create mode 100644 results/n2_sweep.png
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 127.83 KiB | 9.83 MiB/s, done.
Total 7 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.co

In [ ]:
# Sync notebook to repo and Drive
import json
from google.colab import _message

nb_data = _message.blocking_request('get_ipynb')['ipynb']
if 'metadata' not in nb_data:
    nb_data['metadata'] = {}
if 'colab' not in nb_data['metadata']:
    nb_data['metadata']['colab'] = {}
nb_data['metadata']['colab']['name'] = "Quantum-Variational-Eigensolver.ipynb"

repo_nb_path = "/content/Variational-Quantum-Eigensolver/Quantum-Variational-Eigensolver.ipynb"
with open(repo_nb_path, "w") as f:
    json.dump(nb_data, f, indent=2)

drive_nb_path = "/content/drive/MyDrive/mol-simul-VQE/Quantum-Variational-Eigensolver.ipynb"
with open(drive_nb_path, "w") as f:
    json.dump(nb_data, f, indent=2)

colab_nb_drive = "/content/drive/MyDrive/Colab Notebooks/Quantum-Variational-Eigensolver.ipynb"
with open(colab_nb_drive, "w") as f:
    json.dump(nb_data, f, indent=2)

print("Saved cleanly to repo and Drive!")
